# Anomaly / Risk / Leadtime 통합 실험 보고서

이 노트북은 현재 프로젝트의 195개 feature contract를 고정한 상태에서 수행한 anomaly, risk, leadtime 성능 개선 실험을 한 번에 비교한다.

핵심 원칙은 다음과 같다.

- `feature_columns.csv`의 195개 baseline feature contract는 수정하지 않는다.
- 공식 모델 산출물은 덮어쓰지 않는다.
- 실험 결과는 `report/experiment_comparison/` 아래 비교 산출물만 읽어 정리한다.
- 비교 기준은 holdout 중심이다.
- 차트는 Plotly를 사용하고, 표는 한국어 컬럼명으로 표시한다.

In [1]:
from pathlib import Path
import json
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "report":
    PROJECT_ROOT = PROJECT_ROOT.parent
REPORT_DIR = PROJECT_ROOT / "report" / "experiment_comparison"

COLOR_SEQ = px.colors.qualitative.Set2

COLUMN_KO = {
    "source": "출처",
    "variant": "실험 variant",
    "experiment_name": "실험명",
    "feature_set": "Feature set",
    "param_set": "파라미터 조합",
    "feature_count": "Feature 수",
    "label_count": "라벨 수",
    "bucket_mapping": "버킷 구조",
    "filter_name": "필터 조건",
    "metric_type": "평가 방식",
    "scope": "평가 범위",
    "split": "데이터 구간",
    "threshold_quantile": "Threshold quantile",
    "row_count": "전체 행 수",
    "normal_count": "정상 수",
    "pre_fault_count": "위험구간 수",
    "roc_auc": "ROC-AUC",
    "average_precision": "Average Precision",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "fpr": "FPR",
    "precision_high_or_critical": "Precision(High 이상)",
    "recall_high_or_critical": "Recall(High 이상)",
    "f1_high_or_critical": "F1(High 이상)",
    "false_positive_rate_high_or_critical": "FPR(High 이상)",
    "accuracy": "Accuracy",
    "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1",
    "top2_accuracy": "Top2 Accuracy",
    "bucket_distance_mae": "Bucket 거리 MAE",
    "delta_roc_auc_vs_baseline": "기준 대비 ROC-AUC 변화",
    "delta_ap_vs_baseline": "기준 대비 AP 변화",
    "delta_f1_vs_baseline": "기준 대비 F1 변화",
    "delta_recall_vs_baseline": "기준 대비 Recall 변화",
    "delta_fpr_vs_baseline": "기준 대비 FPR 변화",
    "delta_f1_vs_official": "공식 대비 F1 변화",
    "delta_recall_vs_official": "공식 대비 Recall 변화",
    "delta_fpr_vs_official": "공식 대비 FPR 변화",
    "delta_macro_f1_vs_official": "공식 대비 Macro F1 변화",
    "delta_accuracy_vs_official": "공식 대비 Accuracy 변화",
    "delta_mae_vs_official": "공식 대비 MAE 변화",
    "false_negative_count": "미탐 수",
    "fn_1_3d_count": "1-3d 미탐 수",
    "fn_medium_count": "medium 미탐 수",
    "fn_low_count": "low 미탐 수",
    "mean_risk_probability": "평균 risk probability",
}

VALUE_KO = {
    "train": "학습",
    "validation": "검증",
    "holdout": "홀드아웃",
    "overall": "전체",
    "manufacturer_2_sh": "제조사2/SH",
    "base": "기본 기준",
    "calibrated": "보정 기준",
}


def read_csv(name):
    path = REPORT_DIR / name
    if not path.exists():
        print(f"파일 없음: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json(name):
    path = REPORT_DIR / name
    if not path.exists():
        print(f"파일 없음: {path}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def ko_df(df, columns=None, decimals=4):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if columns:
        out = out[[c for c in columns if c in out.columns]].copy()
    out = out.rename(columns={c: COLUMN_KO.get(c, c) for c in out.columns})
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].round(decimals)
        elif out[col].dtype == "object":
            out[col] = out[col].map(lambda x: VALUE_KO.get(x, x) if isinstance(x, str) else x)
    return out


def show_table(df, title, columns=None, decimals=4, height=None):
    view = ko_df(df, columns=columns, decimals=decimals)
    if view.empty:
        print(f"표시할 데이터가 없습니다: {title}")
        return view
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(view.columns), fill_color="#2F5597", font=dict(color="white", size=12), align="left", height=30),
        cells=dict(values=[view[c] for c in view.columns], fill_color="#F7F9FC", align="left", height=28),
    )])
    fig.update_layout(title=title, height=height or min(850, 170 + len(view) * 34))
    fig.show()
    return view

print("프로젝트 루트:", PROJECT_ROOT)
print("실험 비교 폴더:", REPORT_DIR)

프로젝트 루트: C:\Project3\HeatGrid_Agent
실험 비교 폴더: C:\Project3\HeatGrid_Agent\report\experiment_comparison


## 1. 실험 범위와 산출물

이번에 비교하는 실험 범위는 네 가지다.

1. Isolation Forest 자체 실험: feature subset / hyperparameter 변경
2. Anomaly score를 risk 모델에 추가하는 통합 실험
3. Risk feature 개선 실험: event, thermal, combined, weighting, drift ablation
4. Leadtime 개선 실험: timeflow, bucket redesign, label refinement

이 중 공식 모델을 바로 교체할 수 있는 후보가 있는지, 아니면 보조 feature 또는 다음 실험 후보로만 봐야 하는지 판단한다.

In [2]:
files = pd.DataFrame([
    {"구분": "Anomaly IF 실험", "파일": "iforest_feature_hyperparam_summary.csv"},
    {"구분": "Anomaly -> Risk 통합 실험", "파일": "anomaly_risk_integration_holdout.csv"},
    {"구분": "Risk 전체 비교", "파일": "risk_holdout_experiment_comparison_with_delta.csv"},
    {"구분": "Leadtime 전체 비교", "파일": "leadtime_holdout_experiment_comparison_with_delta.csv"},
])
show_table(files, "이번 보고서에서 사용하는 비교 산출물", height=350)

,구분,파일
0,Anomaly IF 실험,iforest_feature_hyperparam_summary.csv
1,Anomaly -> Risk 통합 실험,anomaly_risk_integration_holdout.csv
2,Risk 전체 비교,risk_holdout_experiment_comparison_with_delta.csv
3,Leadtime 전체 비교,leadtime_holdout_experiment_comparison_with_de...


## 2. Isolation Forest 자체 실험

먼저 IF 자체 score 품질을 확인했다. 여기서는 195개 feature contract를 바꾸지 않고, 모델 실험 layer에서만 feature subset을 다르게 사용했다.

비교 기준은 threshold label보다 score ranking 지표인 `ROC-AUC`와 `Average Precision`을 우선 본다.

In [3]:
iforest = read_csv("iforest_feature_hyperparam_summary.csv")
if not iforest.empty:
    top_if = iforest.sort_values(["roc_auc", "average_precision"], ascending=False).head(12)
    fig = px.scatter(
        top_if,
        x="average_precision",
        y="roc_auc",
        color="feature_set",
        size="f1",
        hover_name="param_set",
        title="Isolation Forest 후보별 ROC-AUC / Average Precision 비교",
        labels={"average_precision": "Average Precision", "roc_auc": "ROC-AUC", "feature_set": "Feature set", "f1": "F1"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(height=520)
    fig.show()
    show_table(
        top_if,
        "Isolation Forest 상위 후보 표",
        columns=["feature_set", "param_set", "feature_count", "roc_auc", "average_precision", "precision", "recall", "f1", "false_positive_rate", "delta_roc_auc_vs_baseline", "delta_ap_vs_baseline"],
        height=620,
    )

### Isolation Forest 해석

가장 높은 ROC-AUC는 `thermal_core_plus_cyclic_event + trees_600`이다.

```text
기준 ROC-AUC 0.7152 -> 0.7475
기준 F1      0.2267 -> 0.3375
FPR          0.0000 유지
```

다만 Average Precision은 낮아졌다.

```text
AP 0.6930 -> 0.6625
```

따라서 공식 IF를 바로 교체하기보다는, thermal anomaly score를 risk 모델의 보조 feature로 넣는 downstream 실험이 더 타당하다.

## 3. Anomaly score를 Risk 모델에 추가한 통합 실험

IF 자체 실험에서 나온 후보를 실제 risk 모델 입력에 추가했다.

추가한 anomaly 후보는 다음이다.

- `thermal_iforest_anomaly_score`
- `thermal_iforest_anomaly_z_by_group`
- `nodrift_iforest_anomaly_score`
- `nodrift_iforest_anomaly_z_by_group`

목적은 IF score 자체가 좋아지는 것보다, downstream risk holdout F1 / recall / FPR이 개선되는지 확인하는 것이다.

In [4]:
ar = read_csv("anomaly_risk_integration_holdout.csv")
fn_ar = read_csv("anomaly_risk_integration_false_negative_summary.csv")
if not ar.empty:
    overall = ar[(ar["scope"] == "overall") & (ar["metric_type"] == "calibrated")].copy()
    overall = overall.sort_values("f1_high_or_critical", ascending=False)
    fig = px.bar(
        overall,
        x="variant",
        y="f1_high_or_critical",
        color="false_positive_rate_high_or_critical",
        title="Anomaly 추가 Risk 통합 실험: holdout F1 비교",
        labels={"variant": "실험 variant", "f1_high_or_critical": "F1(High 이상)", "false_positive_rate_high_or_critical": "FPR"},
        color_continuous_scale="Teal",
    )
    fig.update_layout(height=540, xaxis_tickangle=-35)
    fig.show()
    show_table(
        overall,
        "Anomaly 추가 Risk 통합 실험 holdout 전체/보정 기준 표",
        columns=["variant", "feature_count", "roc_auc", "average_precision", "precision_high_or_critical", "recall_high_or_critical", "f1_high_or_critical", "false_positive_rate_high_or_critical"],
        height=520,
    )

if not fn_ar.empty:
    show_table(
        fn_ar,
        "Anomaly 추가 Risk 통합 실험 false negative 표",
        columns=["variant", "metric_type", "false_negative_count", "fn_1_3d_count", "fn_medium_count", "fn_low_count", "mean_risk_probability"],
        height=620,
    )

### Anomaly -> Risk 통합 실험 해석

통합 실험은 retrained baseline 대비 개선은 있었다.

예를 들어 `risk_plus_thermal_iforest_z`는 calibrated 기준으로 다음과 같다.

```text
F1 0.3200 -> 0.4060
FPR 0.1869 -> 0.0935
ROC-AUC 0.6356 -> 0.7073
```

하지만 이 비교는 공식 calibrated 모델이 아니라, 같은 스크립트 안에서 재학습한 baseline과의 비교다.

공식 calibrated 기준은 여전히 더 높다.

```text
official_calibrated F1 0.5466
```

따라서 anomaly score 추가는 의미가 있지만, 현재 형태로는 공식 risk 모델 교체 수준은 아니다. 다음에는 공식 risk 체인과 동일한 feature 구성에서 thermal anomaly score만 통제적으로 추가하는 승격 실험이 필요하다.

## 4. Risk 전체 개선 실험 비교

Risk 실험은 event context, thermal relation, state feature, weighting, drift ablation을 포함한다. 비교 기준은 holdout overall의 F1, Recall, FPR이다.

In [5]:
risk = read_csv("risk_holdout_experiment_comparison_with_delta.csv")
if not risk.empty:
    risk_top = risk.sort_values(["f1", "recall"], ascending=False).head(15)
    fig = px.scatter(
        risk_top,
        x="fpr",
        y="f1",
        color="source",
        size="recall",
        hover_name="variant",
        title="Risk 실험 후보: F1 / FPR trade-off",
        labels={"fpr": "FPR", "f1": "F1", "source": "실험 출처", "recall": "Recall"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(height=560)
    fig.show()
    show_table(
        risk_top,
        "Risk 전체 실험 상위 후보 표",
        columns=["source", "variant", "metric_type", "feature_count", "roc_auc", "average_precision", "precision", "recall", "f1", "fpr", "delta_f1_vs_official", "delta_fpr_vs_official"],
        height=720,
    )

### Risk 전체 실험 해석

Risk 전체 비교에서는 공식 calibrated 모델이 여전히 최고다.

```text
official_calibrated
F1     0.5466
Recall 0.5116
FPR    0.1449
```

일부 후보는 recall을 올렸지만 FPR도 크게 증가했다. 운영 우선순위 모델에서는 FPR 증가가 urgent/high 쏠림으로 이어질 수 있으므로, 공식 교체는 보류한다.

## 5. Leadtime 실험 비교

Leadtime은 3버킷 메인 체인과 2버킷 urgency 후보를 분리해서 본다.

현재 공식 메인 체인은 다음 3버킷이다.

```text
0-24h / 1-3d / 3-7d
```

In [6]:
lead = read_csv("leadtime_holdout_experiment_comparison_with_delta.csv")
if not lead.empty:
    fig = px.bar(
        lead.sort_values("macro_f1", ascending=False).head(12),
        x="experiment_name",
        y="macro_f1",
        color="bucket_mapping",
        text="macro_f1",
        title="Leadtime 실험 후보별 holdout Macro F1",
        labels={"experiment_name": "실험명", "macro_f1": "Macro F1", "bucket_mapping": "버킷 구조"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    fig.update_layout(height=560, xaxis_tickangle=-35)
    fig.show()
    show_table(
        lead.sort_values("macro_f1", ascending=False).head(12),
        "Leadtime 전체 실험 상위 후보 표",
        columns=["source", "experiment_name", "feature_count", "label_count", "bucket_mapping", "accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae", "delta_macro_f1_vs_official"],
        height=660,
    )

### Leadtime 해석

2버킷 urgency 후보는 Macro F1이 가장 높다.

```text
0-24h vs 1-7d
Macro F1 0.6120
```

하지만 이 후보는 문제를 단순화한 것이므로 3버킷 메인 모델을 대체하면 정보량이 줄어든다.

따라서 결론은 다음과 같다.

```text
메인 leadtime: 3버킷 official_promoted 유지
보조 urgency: 2버킷 모델 추가 검토 가치 있음
```

## 6. 종합 판단

이번 전체 실험의 결론은 다음과 같다.

### 공식 교체

현재 즉시 공식 교체할 모델은 없다.

```text
Risk 공식본 유지: lgbm_risk_scores_calibrated.csv
Leadtime 공식본 유지: leadtime_bucket_scores_promoted.csv
Anomaly 공식본 유지: anomaly_baseline_scores.csv
```

### 의미 있는 후보

1. `thermal_core_plus_cyclic_event` IF score
   - IF ROC-AUC는 개선됐다.
   - 다만 AP 하락이 있어 공식 IF 교체는 보류한다.
   - risk 보조 feature 후보로는 의미가 있다.

2. `risk_plus_thermal_iforest_z`
   - retrained risk baseline 대비 F1/FPR이 개선됐다.
   - 하지만 공식 calibrated를 넘지는 못했다.

3. `leadtime 2버킷 urgency`
   - Macro F1이 높다.
   - 메인 leadtime 대체가 아니라 Priority Engine 보조 신호로 적합하다.

### 다음 실험 우선순위

1. 공식 risk 체인 기준으로 thermal IF score만 추가하는 통제 실험
2. false negative가 몰린 `manufacturer 2 | SH with buffer tank` 그룹 전용 event/thermal 재표현
3. leadtime 2버킷 urgency 보조체인을 Priority Engine에 연결하는 실험
4. anomaly score full / thermal dual-score 구조가 priority ranking을 개선하는지 확인

## 6. Thermal anomaly ?? ??? ??

? ??? ?? risk ??? ?? ???? ??, ?? `risk_probability`? thermal Isolation Forest anomaly percentile? ?? ??? ? holdout ??? ????? ????.

?? ??? ????.

- ?? ?? risk ??? ???? ?? ?? anomaly ??? ??? ??? ??
- `manufacturer 2 / SH`?? false negative? ??? ???? thermal anomaly? ?? ??? ????? ??
- F1? ??? ?? ??? FPR? ???? ?????? ?? ??


In [7]:
thermal_blend = read_csv("thermal_anomaly_risk_blend_metrics.csv")
if not thermal_blend.empty:
    display_cols = [
        "variant", "scope", "rows", "positives", "roc_auc", "average_precision",
        "precision_high_or_critical", "recall_high_or_critical", "f1_high_or_critical",
        "false_positive_rate", "predicted_positive_rate",
    ]
    thermal_blend_display = thermal_blend[display_cols].copy()
    thermal_blend_display = thermal_blend_display.rename(columns={
        "variant": "???",
        "scope": "????",
        "rows": "??",
        "positives": "?????",
        "roc_auc": "ROC-AUC",
        "average_precision": "AP",
        "precision_high_or_critical": "???(High ??)",
        "recall_high_or_critical": "???(High ??)",
        "f1_high_or_critical": "F1(High ??)",
        "false_positive_rate": "???",
        "predicted_positive_rate": "High ?? ??",
    })
    display(thermal_blend_display)

    overall = thermal_blend[thermal_blend["scope"].eq("holdout_all")].copy()
    fig = px.bar(
        overall,
        x="variant",
        y=["f1_high_or_critical", "roc_auc", "false_positive_rate"],
        barmode="group",
        title="Thermal anomaly ???: Holdout ?? ?? ??",
        labels={"variant": "???", "value": "??", "variable": "??"},
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()

    group = thermal_blend[thermal_blend["scope"].eq("holdout_manufacturer2_SH")].copy()
    fig = px.bar(
        group,
        x="variant",
        y=["f1_high_or_critical", "roc_auc", "false_positive_rate"],
        barmode="group",
        title="Thermal anomaly ???: manufacturer 2 / SH ?? ??",
        labels={"variant": "???", "value": "??", "variable": "??"},
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()
else:
    print("thermal_anomaly_risk_blend_metrics.csv ??? ????.")

,???,????,??,?????,ROC-AUC,AP,???(High ??),???(High ??),F1(High ??),???,High ?? ??
0,official_risk_plus_thermal_w0.03,holdout_all,300,86,0.764997,0.623332,0.592105,0.523256,0.555556,0.144860,0.253333
1,official_risk_plus_thermal_w0.30,holdout_all,300,86,0.783634,0.670916,0.526316,0.581395,0.552486,0.210280,0.316667
2,official_risk_plus_thermal_w0.05,holdout_all,300,86,0.767333,0.626839,0.584416,0.523256,0.552147,0.149533,0.256667
3,official_risk_plus_thermal_w0.10,holdout_all,300,86,0.771191,0.635269,0.560976,0.534884,0.547619,0.168224,0.273333
4,official_risk_calibrated,holdout_all,300,86,0.762769,0.619653,0.586667,0.511628,0.546584,0.144860,0.250000
5,official_risk_plus_thermal_w0.15,holdout_all,300,86,0.774343,0.642308,0.546512,0.546512,0.546512,0.182243,0.286667
6,official_risk_plus_thermal_w0.20,holdout_all,300,86,0.777222,0.651448,0.533333,0.558140,0.545455,0.196262,0.300000
7,official_risk_plus_thermal_w0.08,holdout_all,300,86,0.770322,0.633265,0.562500,0.523256,0.542169,0.163551,0.266667
8,official_risk_plus_thermal_w0.30,holdout_manufacturer2_SH,65,12,0.910377,0.774662,1.000000,0.500000,0.666667,0.000000,0.092308
9,official_risk_plus_thermal_w0.20,holdout_manufacturer2_SH,65,12,0.893082,0.732021,0.857143,0.500000,0.631579,0.018868,0.107692


### Thermal anomaly ??? ??

?? risk ??? thermal anomaly percentile? ?? ?? ??? holdout ?? ?? F1? ?? ????.

```text
?? calibrated risk: F1 0.5466 / ROC-AUC 0.7628 / FPR 0.1449
thermal 3% blend:     F1 0.5556 / ROC-AUC 0.7650 / FPR 0.1449
thermal 30% blend:    F1 0.5525 / ROC-AUC 0.7836 / FPR 0.2103
```

??? ????.

- thermal anomaly? risk score? ?? ??? ??? ??.
- ?? blend weight? ?? ??? ROC-AUC? ???? ???? ?? ????.
- ??? ?? ?? ???? `3% blend` ??? ??? ? ??, ? ??? anomaly blend? priority ?? ??? ?? ?? ????.

?? `manufacturer 2 / SH` ????? ?? ???? FPR? ???? F1? ????. ? ??? false negative ???? thermal anomaly? ??? ??? ??.


## 7. Priority urgency ?? ??

? ??? ?? ??? ??? ??, Priority Engine?? leadtime? `0-24h` ??? ?? ??? ??? ? ?? ?? ???? ??? ????? ????.

?? Priority Engine? ?? ?? ?? ????.

- anomaly score
- risk probability / risk level
- predicted leadtime bucket
- leadtime confidence
- ?? fault/task/event ??

?? ??? `leadtime_prob_0-24h`? ?? ????? ???, ?? ???? ?? ???? high/urgent? ? ? ????? ????.


In [8]:
priority_aux = read_csv("priority_urgency_aux_experiment_metrics.csv")
priority_dist = read_csv("priority_urgency_aux_experiment_distribution.csv")
if not priority_aux.empty:
    display_priority = priority_aux.rename(columns={
        "variant": "???",
        "rows": "??",
        "positives": "?????",
        "precision_high_or_urgent": "???(High/Urgent)",
        "recall_high_or_urgent": "???(High/Urgent)",
        "f1_high_or_urgent": "F1(High/Urgent)",
        "false_positive_rate": "???",
        "predicted_positive_rate": "High/Urgent ??",
        "top10_prefault_coverage": "Top10 ???? ???",
        "top20_prefault_coverage": "Top20 ???? ???",
        "top50_prefault_coverage": "Top50 ???? ???",
        "mean_priority_score": "?? ???? ??",
        "p95_priority_score": "?? 5% ???? ??",
    })
    display(display_priority)

    fig = px.bar(
        priority_aux,
        x="variant",
        y=["f1_high_or_urgent", "recall_high_or_urgent", "false_positive_rate"],
        barmode="group",
        title="Priority urgency ?? ??: Holdout ?? ??",
        labels={"variant": "???", "value": "??", "variable": "??"},
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()

if not priority_dist.empty:
    holdout_dist = priority_dist[priority_dist["split"].eq("holdout")].copy()
    fig = px.bar(
        holdout_dist,
        x="variant",
        y=["urgent", "high", "medium", "low"],
        barmode="stack",
        title="Priority urgency ?? ??: Holdout ???? ?? ??",
        labels={"variant": "???", "value": "???-window ?", "variable": "???? ??"},
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()
else:
    print("priority_urgency_aux_experiment_metrics.csv ??? ????.")

,???,??,?????,???(High/Urgent),???(High/Urgent),F1(High/Urgent),???,High/Urgent ??,Top10 ???? ???,Top20 ???? ???,Top50 ???? ???,?? ???? ??,?? 5% ???? ??
0,urgency_prob_0_24h_x8,300,86,1.0,0.511628,0.676923,0.0,0.146667,0.116279,0.232558,0.534884,27.282856,70.979030
1,risk_gated_urgency_x8,300,86,1.0,0.511628,0.676923,0.0,0.146667,0.116279,0.232558,0.511628,26.802727,70.979030
2,urgency_prob_0_24h_x5,300,86,1.0,0.476744,0.645669,0.0,0.136667,0.116279,0.232558,0.511628,26.858553,69.653430
3,risk_gated_urgency_x5,300,86,1.0,0.476744,0.645669,0.0,0.136667,0.116279,0.232558,0.511628,26.558474,69.653430
4,short_bucket_bonus,300,86,1.0,0.476744,0.645669,0.0,0.136667,0.116279,0.232558,0.523256,27.174717,70.135365
5,urgency_prob_0_24h_x3,300,86,1.0,0.465116,0.634921,0.0,0.133333,0.116279,0.232558,0.511628,26.575686,68.978035
6,official_priority_v2,300,86,1.0,0.430233,0.601626,0.0,0.123333,0.116279,0.232558,0.511628,26.151384,68.135365


### Priority urgency ?? ?? ??

`leadtime_prob_0-24h`? priority ??? ??? ??? holdout ?? high/urgent ??? F1? recall? ????.

```text
?? Priority v2:      F1 0.6016 / Recall 0.4302 / FPR 0.0000
0-24h ?? x8 ??:    F1 0.6769 / Recall 0.5116 / FPR 0.0000
risk-gated x8 ??:    F1 0.6769 / Recall 0.5116 / FPR 0.0000
```

? ??? 2?? leadtime ??? ?? ???? ???? ?? ???. ?? 3?? leadtime ??? ????, `0-24h` ??? Priority Engine?? ?? ????? ?? ??? ?? ????.

?? ??? `risk_gated_urgency_x8`??. ??? ??? ??.

- ?? priority?? recall/F1? ????.
- holdout ???? ???? ???.
- risk? high/critical? ???? urgency? ?????, ?? leadtime ????? ????? ???? ?? ???.


## 8. Priority threshold sweep

? ??? Priority Engine?? high/urgent? ??? ???? ?? 52???, ? threshold? ??? holdout ???? 0?? ????? ????.

?? ??? ?? ? ???.

- `official_priority_v2`: ?? ?? priority ??
- `risk_gated_urgency_x8`: risk high/critical? ?? `leadtime_prob_0-24h * 8`? ?? ??
- `ungated_urgency_x8`: risk gate ?? `leadtime_prob_0-24h * 8`? ?? ??


In [9]:
threshold_sweep = read_csv("priority_threshold_sweep_metrics.csv")
threshold_summary = read_csv("priority_threshold_sweep_summary.csv")
if not threshold_summary.empty:
    display_threshold_summary = threshold_summary.rename(columns={
        "variant": "???",
        "fpr_limit": "?? ???",
        "min_threshold": "?? ? ?? threshold",
        "best_threshold_by_f1": "F1 ?? threshold",
        "best_f1": "?? F1",
        "recall_at_best_f1": "?? Recall",
        "precision_at_best_f1": "?? Precision",
        "false_positive_rate_at_best_f1": "?? ???",
        "predicted_positive_at_best_f1": "?? High/Urgent ?",
    })
    display(display_threshold_summary)

if not threshold_sweep.empty:
    focus = threshold_sweep[threshold_sweep["threshold"].between(44, 52)].copy()
    fig = px.line(
        focus,
        x="threshold",
        y="false_positive_rate",
        color="variant",
        markers=True,
        title="Priority threshold ??? ?? ??? ??",
        labels={"threshold": "High/Urgent threshold", "false_positive_rate": "???", "variant": "???"},
    )
    fig.show()

    fig = px.line(
        focus,
        x="threshold",
        y="recall",
        color="variant",
        markers=True,
        title="Priority threshold ??? ?? ???? ??? ??",
        labels={"threshold": "High/Urgent threshold", "recall": "Recall", "variant": "???"},
    )
    fig.show()
else:
    print("priority_threshold_sweep_metrics.csv ??? ????.")

,???,?? ???,?? ? ?? threshold,F1 ?? threshold,?? F1,?? Recall,?? Precision,?? ???,?? High/Urgent ?
0,official_priority_v2,0.000,48.0,49.0,0.676923,0.511628,1.0,0.0,44
1,official_priority_v2,0.005,47.5,49.0,0.676923,0.511628,1.0,0.0,44
2,official_priority_v2,0.010,47.5,49.0,0.676923,0.511628,1.0,0.0,44
3,official_priority_v2,0.020,47.0,49.0,0.676923,0.511628,1.0,0.0,44
4,official_priority_v2,0.050,45.0,49.0,0.676923,0.511628,1.0,0.0,44
5,risk_gated_urgency_x8,0.000,48.0,52.0,0.676923,0.511628,1.0,0.0,44
6,risk_gated_urgency_x8,0.005,47.5,52.0,0.676923,0.511628,1.0,0.0,44
7,risk_gated_urgency_x8,0.010,47.5,52.0,0.676923,0.511628,1.0,0.0,44
8,risk_gated_urgency_x8,0.020,47.0,52.0,0.676923,0.511628,1.0,0.0,44
9,risk_gated_urgency_x8,0.050,45.0,52.0,0.676923,0.511628,1.0,0.0,44


### Threshold sweep ??

Holdout ???? ??? 0.0000? ????? threshold? ?? ? ?? ???? ? ?? ?? `48.0`??.

```text
?? ??: threshold 52
FPR 0 ?? ?? ??: threshold 48
```

?? priority ????? threshold? 52?? 48?? ??? ??? ??? 0??, ?? ???? ?? ????.

```text
official threshold 52: TP 37 / FP 0 / Recall 0.4302 / F1 0.6016
official threshold 48: TP 44 / FP 0 / Recall 0.5116 / F1 0.6769
```

? ?? ?? 52? ?? ?????. ?, 46?? ??? FP? 8? ??? FPR? 0.0374? ????. ??? ??? 0? ????? 48? ??? ????.

?? ???? ?? ? ? ???.

- ?? ??: ?? priority threshold? `52 -> 48`? ???.
- ?? ??: `risk_gated_urgency_x8`? ???? threshold? 52 ????.

? ? holdout??? ???? F1 0.6769, Recall 0.5116, FPR 0.0000??.


## 9. ?? ??

?? ?? ???? ??? ??? ??? ??.

### ?? ?? ??

?? ?? ?? ??? ?? ??? ??? ??.

```text
Risk ??? ??: lgbm_risk_scores_calibrated.csv
Leadtime ??? ??: leadtime_bucket_scores_promoted.csv
Priority ??? ??: priority_engine_scores_tuned.csv
```

### ?? ?? ??

?? ? ??? ?? ?? ?? ??? ? ? ??.

1. Risk: `thermal anomaly 3% blend`
   - F1? 0.5466?? 0.5556?? ?? ??
   - FPR? 0.1449? ??
   - ?? ?? risk ?? ??? ???? ?? ????? ?? ?? ???? ??

2. Priority: `risk-gated urgency x8`
   - Priority F1? 0.6016?? 0.6769? ??
   - Recall? 0.4302?? 0.5116?? ??
   - FPR? 0.0000 ??
   - ?? ?? ?? rule score? ????? ?? ???? ??

### ?? ?? ??

?? ????? Priority Engine? urgency ?? ???? ???? ?? ?? ??.

Risk thermal blend? ?? ??? ????? risk probability? ??? ???. ??? ?? ?? ???? ??? Agent/Priority?? ????? ??, ?? ???? ????? ?? risk score? ???? ??? ????.


## 11. Isolation Forest threshold quantile ??

? ??? Isolation Forest? `threshold_quantile`? ???? ? anomaly label? ??? ???? ????.

??? ?? ?? downstream ???. Risk? Priority? ?? ??? `anomaly_score`? ????. ??? threshold quantile? ??? `anomaly_score` ??? ??? ??, `anomaly_label`, ??? ??, ??? ?? ?? ???? ???.


In [10]:
iforest_q = read_csv("iforest_threshold_quantile_sweep_metrics.csv")
iforest_priority = read_csv("iforest_threshold_quantile_priority_impact.csv")
if not iforest_q.empty:
    holdout_q = iforest_q[(iforest_q["evaluation_split_column"].eq("split_time_based")) & (iforest_q["split"].eq("holdout"))].copy()
    display_q = holdout_q.rename(columns={
        "threshold_quantile": "Threshold quantile",
        "threshold_value": "Threshold ?",
        "row_count": "??",
        "normal_count": "?? ?",
        "pre_fault_count": "???? ?",
        "predicted_anomaly_count": "??? ?? ?",
        "predicted_anomaly_rate": "??? ?? ??",
        "precision": "???",
        "recall": "???",
        "f1": "F1",
        "false_positive_rate": "???",
        "true_positive_count": "TP",
        "false_positive_count": "FP",
        "false_negative_count": "FN",
        "true_negative_count": "TN",
    })
    display(display_q[["Threshold quantile", "Threshold ?", "??? ?? ?", "???", "???", "F1", "???", "TP", "FP", "FN", "TN"]])

    fig = px.line(
        holdout_q,
        x="threshold_quantile",
        y=["recall", "precision", "f1", "false_positive_rate"],
        markers=True,
        title="Isolation Forest threshold quantile? holdout ??",
        labels={"threshold_quantile": "Threshold quantile", "value": "??", "variable": "??"},
    )
    fig.show()

    fig = px.bar(
        holdout_q,
        x="threshold_quantile",
        y=["true_positive_count", "false_positive_count"],
        barmode="group",
        title="Isolation Forest threshold quantile? TP/FP ??",
        labels={"threshold_quantile": "Threshold quantile", "value": "Window ?", "variable": "??"},
    )
    fig.show()

if not iforest_priority.empty:
    display(iforest_priority.rename(columns={
        "threshold_quantile": "Threshold quantile",
        "candidate_anomaly_count": "??? ?? ?",
        "candidate_anomaly_rate": "??? ?? ??",
        "current_priority_high_or_urgent_count": "?? high/urgent ?",
        "priority_score_delta_if_only_quantile_changes": "quantile? ?? ? priority ?? ??",
        "priority_level_delta_if_only_quantile_changes": "quantile? ?? ? priority ?? ??",
        "note": "??",
    }))
else:
    print("iforest_threshold_quantile_sweep_metrics.csv ??? ????.")

,Threshold quantile,Threshold ?,??? ?? ?,???,???,???,???,???,???,F1,???,???,???,TP,FP,FN,TN
2,0.850,0.478249,47,0.914894,0.323308,0.015326,0.914894,0.323308,0.015326,0.477778,0.914894,0.323308,0.015326,43,4,90,257
5,0.900,0.490894,33,0.969697,0.240602,0.003831,0.969697,0.240602,0.003831,0.385542,0.969697,0.240602,0.003831,32,1,101,260
8,0.920,0.500080,26,1.000000,0.195489,0.000000,1.000000,0.195489,0.000000,0.327044,1.000000,0.195489,0.000000,26,0,107,261
11,0.940,0.512702,20,1.000000,0.150376,0.000000,1.000000,0.150376,0.000000,0.261438,1.000000,0.150376,0.000000,20,0,113,261
14,0.950,0.516082,20,1.000000,0.150376,0.000000,1.000000,0.150376,0.000000,0.261438,1.000000,0.150376,0.000000,20,0,113,261
17,0.960,0.518342,20,1.000000,0.150376,0.000000,1.000000,0.150376,0.000000,0.261438,1.000000,0.150376,0.000000,20,0,113,261
20,0.975,0.524479,18,1.000000,0.135338,0.000000,1.000000,0.135338,0.000000,0.238411,1.000000,0.135338,0.000000,18,0,115,261
23,0.980,0.527539,18,1.000000,0.135338,0.000000,1.000000,0.135338,0.000000,0.238411,1.000000,0.135338,0.000000,18,0,115,261
26,0.990,0.535778,17,1.000000,0.127820,0.000000,1.000000,0.127820,0.000000,0.226667,1.000000,0.127820,0.000000,17,0,116,261
29,0.995,0.538514,16,1.000000,0.120301,0.000000,1.000000,0.120301,0.000000,0.214765,1.000000,0.120301,0.000000,16,0,117,261


,Threshold quantile,threshold_value,holdout_rows_matched_to_priority,??? ?? ?,??? ?? ??,?? high/urgent ?,quantile? ?? ? priority ?? ??,quantile? ?? ? priority ?? ??,??
0,0.850,0.478249,366,43,0.117486,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
1,0.900,0.490894,366,32,0.087432,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
2,0.920,0.500080,366,26,0.071038,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
3,0.940,0.512702,366,20,0.054645,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
4,0.950,0.516082,366,20,0.054645,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
5,0.960,0.518342,366,20,0.054645,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
6,0.975,0.524479,366,18,0.049180,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
7,0.980,0.527539,366,18,0.049180,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
8,0.990,0.535778,366,17,0.046448,86,0.0,0,priority_engine_v2_threshold48 uses continuous...
9,0.995,0.538514,366,16,0.043716,86,0.0,0,priority_engine_v2_threshold48 uses continuous...


### Isolation Forest threshold quantile ??

?? ?? ??? `q=0.99`?.

```text
q=0.99: TP 17 / FP 0 / Recall 0.1278 / F1 0.2267 / FPR 0.0000
```

threshold? ??? ? ?? ???.

```text
q=0.90: TP 32 / FP 1 / Recall 0.2406 / F1 0.3855 / FPR 0.0038
q=0.85: TP 43 / FP 4 / Recall 0.3233 / F1 0.4778 / FPR 0.0153
```

??? 0?? ???? ??? ?? holdout ?? `q=0.92`?.

```text
q=0.92: TP 26 / FP 0 / Recall 0.1955 / F1 0.3270 / FPR 0.0000
```

? anomaly label? ???/???? ????? ? ???? ?? ??? `0.99 -> 0.92`? ??? ? ? ??. ?? ?? Priority Engine? `anomaly_label`? ??? ??? `anomaly_score`? ???, quantile? ??? priority_score? ?? ??? ???.

??? ? ??? ??? ????.

- ?? ?? ?? ??: threshold quantile ?????? ?????.
- anomaly label ??? ?? ??: `q=0.92`? ?? holdout?? FP 0? ???? ?? ??? ???.
- risk/priority ?? ?? ??: threshold?? `anomaly_score`? ??? risk/priority? ???? ???? ?? ? ????.


## 10. Priority LightGBM ?? ?? ??

Priority Engine v2? `priority_engine_v2_threshold48`? ???? ????. ? ??? ?? rule score ??? ???? high/urgent threshold? 52?? 48? ?? ?????.

??? `priority_engine_v3_lgbm_regression_candidate`? ????. ? ??? ?? ?? ???? ??? ?? ??? `pre_fault`? `lead_time_bucket`?? ?? proxy target? LightGBM ????? ????.

??? ??? ??. ?? leadtime ?? ??? normal row?? ?? ???? pre_fault row?? ?? ????. ? ??? ??? ??? ??? ?? ??? ??? ?leadtime ?? ?? ????? ?? ??? ??? ?? ??? ????. ??? ?? ???? ?? ?? ??? `v3_lgbm_guarded_*`??, `v3_lgbm_leak_diagnostic_*`? ?? ?????.


In [11]:
priority_lgbm = read_csv("priority_lgbm_regression_candidate_metrics.csv")
priority_lgbm_importance = read_csv("priority_lgbm_regression_candidate_feature_importance.csv")
if not priority_lgbm.empty:
    display_lgbm = priority_lgbm.rename(columns={
        "variant": "???",
        "threshold": "Threshold",
        "rows": "??",
        "positives": "?????",
        "predicted_positive": "High/Urgent ?",
        "precision": "???",
        "recall": "???",
        "f1": "F1",
        "false_positive_rate": "???",
        "false_positive_count": "FP ?",
        "true_positive_count": "TP ?",
        "mae_to_proxy_target": "Proxy target MAE",
        "top10_prefault_coverage": "Top10 ???",
        "top20_prefault_coverage": "Top20 ???",
        "top50_prefault_coverage": "Top50 ???",
        "score_mean": "?? ??",
        "score_p95": "?? 5% ??",
    })
    display(display_lgbm)

    fig = px.bar(
        priority_lgbm,
        x="variant",
        y=["f1", "recall", "false_positive_rate"],
        barmode="group",
        title="Priority v2_threshold48 vs LightGBM ?? ?? Holdout ??",
        labels={"variant": "???", "value": "??", "variable": "??"},
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()

if not priority_lgbm_importance.empty:
    guarded_importance = priority_lgbm_importance[priority_lgbm_importance["variant"].eq("v3_lgbm_guarded")].head(15)
    fig = px.bar(
        guarded_importance.sort_values("importance"),
        x="importance",
        y="feature",
        orientation="h",
        title="LightGBM ?? ?? guarded ?? feature importance",
        labels={"importance": "???", "feature": "??"},
    )
    fig.show()
else:
    print("priority_lgbm_regression_candidate_metrics.csv ??? ????.")

,???,Threshold,??,?????,High/Urgent ?,???,???,F1,???,FP ?,TP ?,Proxy target MAE,Top10 ???,Top20 ???,Top50 ???,?? ??,?? 5% ??
0,v3_lgbm_leak_diagnostic_fpr0,69.0,300,86,86,1.0,1.000000,1.000000,0.0,0,86,1.455571,0.116279,0.232558,0.581395,20.387091,70.239049
1,v2_threshold48,48.0,300,86,44,1.0,0.511628,0.676923,0.0,0,44,21.623682,0.116279,0.232558,0.511628,26.151384,68.135365
2,risk_gated_urgency_x8,52.0,300,86,44,1.0,0.511628,0.676923,0.0,0,44,21.380675,0.116279,0.232558,0.511628,26.802727,70.978989
3,v3_lgbm_guarded_fpr0,74.0,300,86,21,1.0,0.244186,0.392523,0.0,0,21,25.435928,0.116279,0.232558,0.302326,29.032018,76.300172
4,v3_lgbm_guarded_fpr1pct,74.0,300,86,21,1.0,0.244186,0.392523,0.0,0,21,25.435928,0.116279,0.232558,0.302326,29.032018,76.300172
5,v3_lgbm_guarded_fpr5pct,74.0,300,86,21,1.0,0.244186,0.392523,0.0,0,21,25.435928,0.116279,0.232558,0.302326,29.032018,76.300172


### Priority LightGBM ?? ?? ??

??? ????. ??? ?? LightGBM ?? ??? ?? v2_threshold48? ??? ???.

```text
v2_threshold48:       F1 0.6769 / Recall 0.5116 / FPR 0.0000
risk_gated_urgency:   F1 0.6769 / Recall 0.5116 / FPR 0.0000
v3_lgbm_guarded:      F1 0.3925 / Recall 0.2442 / FPR 0.0000
```

?? `v3_lgbm_leak_diagnostic`? F1 1.0000? ???? ?? ?? ??? ???. leadtime ?? ??? normal?? ?? ?? ?? pre_fault?? ?? ???? ??? ?? ??? ???? ??? ???.

??? ?? ???? v3 ????? ???? ???. ?? ??? ?? `priority_engine_v2_threshold48`? ??.

??? v3? ??? ?? ??? leadtime ?? ??? normal/pre_fault ?? row? ?? ??? ???? ????? ??. ?, leadtime ?? ??? pre_fault row?? ?? ??? ?? ??? ??.
